In [ ]:
!pip install -q openai-whisper opencv-python-headless librosa soundfile torch

In [ ]:
import cv2, numpy as np, librosa, subprocess, json, glob, torch, whisper
from pathlib import Path
from google.colab import files

In [ ]:
print("Пожалуйста, загрузите видеофайл...")
uploaded = files.upload()

In [ ]:
def extract_audio(video_path, output_wav, target_sr=16000):
    subprocess.run(['ffmpeg', '-i', str(video_path), '-vn', '-acodec', 'pcm_s16le',
                    '-ar', str(target_sr), '-ac', '1', '-y', str(output_wav)], check=True)
    print(f"Аудио извлечено: {output_wav}")

In [ ]:
def transcribe_audio(audio_path, output_json, model_size="base", language="zh"):
    model = whisper.load_model(model_size)
    result = model.transcribe(str(audio_path), language=language, word_timestamps=True)
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Транскрипция завершена: {output_json}")
    return result

In [ ]:
def extract_gesture_keypoints(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    all_kp, timestamps = [], []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        timestamps.append(frame_idx / fps)
        # Детекция цвета кожи
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        lower = np.array([0, 20, 70], dtype=np.uint8)
        upper = np.array([20, 255, 255], dtype=np.uint8)
        mask = cv2.inRange(hsv, lower, upper)
        mask = cv2.erode(mask, None, iterations=2)
        mask = cv2.dilate(mask, None, iterations=2)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        kp = np.zeros((21, 3), dtype=np.float32)  # 21 ключевая точка, каждая (x,y,z)
        if contours:
            # Берём самый большой контур
            c = max(contours, key=cv2.contourArea)
            M = cv2.moments(c)
            if M["m00"] != 0:
                cx = M["m10"] / M["m00"]
                cy = M["m01"] / M["m00"]
                # Координаты центра ладони (нормализованные)
                kp[0] = [cx / frame.shape[1], cy / frame.shape[0], 0.0]
                # Выпуклая оболочка
                hull = cv2.convexHull(c)
                hull_pts = hull.reshape(-1, 2)  # (H, 2)
                if len(hull_pts) > 0:
                    # Вычисляем расстояния от точек оболочки до центра ладони, берём 4 самые дальние как кончики пальцев
                    dists = np.linalg.norm(hull_pts - np.array([cx, cy]), axis=1)
                    top4_idx = np.argsort(dists)[-4:][::-1] # индексы 4 самых дальних
                    for i, idx in enumerate(top4_idx):
                        pt = hull_pts[idx]
                        kp[i+1] = [pt[0] / frame.shape[1], pt[1] / frame.shape[0], 0.0]
        all_kp.append(kp)
        frame_idx += 1
    cap.release()
    print(f"Ключевые точки жестов: всего{len(all_kp)} кадров")
    return np.array(all_kp, dtype=np.float32), np.array(timestamps, dtype=np.float32)


In [ ]:
def extract_prosody_features(audio_path, hop_ms=10):
    y, sr = librosa.load(audio_path, sr=None)
    hop = int(sr * hop_ms / 1000)
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)
    f0, _, _ = librosa.pyin(y, fmin=75, fmax=400, sr=sr, hop_length=hop)
    f0 = np.nan_to_num(f0, nan=0.0)
    return times, f0, rms

In [ ]:
class SimpleAlignmentDataset:
    def __init__(self, g_kp, g_ts, segs, p_times, p_feat, window=1.0, neg_ratio=1):
        self.g_ts = g_ts
        self.g_kp = g_kp.reshape(len(g_kp), -1)  # (N, 63)
        self.p_times = p_times
        self.p_feat = np.column_stack([p_feat[0][:, None], p_feat[1][:, None]])  # (M,2)
        self.segs = segs
        self.window = window
        self.neg_ratio = neg_ratio
        self._build_pairs()

    def _extract_window(self, ts, feat, center):
        start = center - self.window/2
        end = center + self.window/2
        idx = np.where((ts >= start) & (ts < end))[0]
        if len(idx) == 0:
            return None
        window = feat[idx]
        target_len = 32
        if len(window) >= target_len:
            indices = np.linspace(0, len(window)-1, target_len, dtype=int)
        else:
            indices = np.arange(len(window))
            indices = np.pad(indices, (0, target_len - len(indices)), constant_values=indices[-1])
        return window[indices]

    def _build_pairs(self):
        self.pos_pairs = []
        for seg in self.segs:
            center = (seg['start'] + seg['end']) / 2
            g_win = self._extract_window(self.g_ts, self.g_kp, center)
            p_win = self._extract_window(self.p_times, self.p_feat, center)
            if g_win is not None and p_win is not None:
                self.pos_pairs.append((g_win, p_win))
        all_p = [p for _, p in self.pos_pairs]
        self.neg_pairs = []
        for _ in range(len(self.pos_pairs) * self.neg_ratio):
            i = np.random.randint(len(self.pos_pairs))
            j = np.random.randint(len(all_p))
            self.neg_pairs.append((self.pos_pairs[i][0], all_p[j]))

    def get_samples(self):
        samples = [(torch.FloatTensor(g), torch.FloatTensor(p), 1) for g, p in self.pos_pairs]
        samples += [(torch.FloatTensor(g), torch.FloatTensor(p), 0) for g, p in self.neg_pairs]
        return samples

In [ ]:
Path("./temp").mkdir(exist_ok=True)
videos = glob.glob("*.mp4") + glob.glob("*.avi") + glob.glob("*.mov")
if not videos:
    print("Видеофайл не найден! Пожалуйста, запустите ячейку заново и загрузите видео.")
else:
    v = Path(videos[0])
    print(f"\n=== Обработка видео: {v.name} ===\n")
    # Аудио
    audio_path = Path("./temp") / f"{v.stem}.wav"
    extract_audio(v, audio_path)
    # Транскрипция Whisper
    whisper_json = Path("./temp") / f"{v.stem}.json"
    whisper_result = transcribe_audio(audio_path, whisper_json)
    # Жесты
    g_kp, g_ts = extract_gesture_keypoints(v)
    # Просодия
    p_times, f0, energy = extract_prosody_features(audio_path)
    # Набор данных
    dataset = SimpleAlignmentDataset(g_kp, g_ts, whisper_result["segments"], p_times, (energy, f0))
    samples = dataset.get_samples()
    torch.save(samples, "./multimodal_dataset.pt")
    print(f"\n Готово! Создано {len(samples)} примеров (положительных: {len(dataset.pos_pairs)}, отрицательных: {len(dataset.neg_pairs)})")
    print(f"   Набор данных сохранён как: ./multimodal_dataset.pt")
    print(f"   Пример: форма жестов {samples[0][0].shape}, форма просодии {samples[0][1].shape}, метка {samples[0][2]}")

    # Очистка временных файлов
    import shutil
    shutil.rmtree("./temp", ignore_errors=True)